In [ ]:
import json

with open("/content/drive/MyDrive/Colab Notebooks/test.jsonl", "r") as f:
    for line in f:
        obj = json.loads(line)
        #print(obj)

In [1]:
!mkdir onnx-out
!mkdir awq-out
!mkdir temp

In [2]:
#!git clone https://huggingface.co/onnx-community/Qwen3-0.6B-ONNX
!git clone https://huggingface.co/FreedomIntelligence/Apollo2-1.5B
import os
os.rename("./Apollo2-1.5B", "./Apollo2")

Cloning into 'Apollo2-1.5B'...
remote: Enumerating objects: 52, done.
remote: Total 52 (delta 0), reused 0 (delta 0), pack-reused 52 (from 1)
Receiving objects: 100% (52/52), 3.62 MiB | 8.77 MiB/s, done.
Resolving deltas: 100% (18/18), done.


In [3]:
#!git clone https://github.com/casper-hansen/AutoAWQ
#!cd AutoAWQ
#!pip install -e .
!pip install autoawq
!pip install onnx_ir==0.2.0
!pip install onnxruntime-genai==0.12.0
#!pip install onnxruntime-genai-cuda==0.12.0
!pip install transformers==4.51.3 -U

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for autoawq: filename=autoawq-0.2.9-py3-none-any.whl size=115106 sha256=571c5c4a32d830aa47109a2162d8bd2971da29fb7478e303109f05edece93189
  Stored in directory: /root/.cache/pip/wheels/45/1a/7b/7314b3a958454e8ce349f600829a3f0a6a05aeebf987be1e16
Successfully built autoawq
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 243.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 208.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 157.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface

In [5]:
from datasets import load_dataset
data = load_dataset(
    "wikitext",'wikitext-103-raw-v1',streaming=True)["train"]

In [6]:
calibration_texts_iterator = data["text"]
calibration_texts = []
# Collect a limited number of samples from the streaming dataset for calibration
# Adjust 1000 to your desired number of calibration samples
for i, text_item in enumerate(calibration_texts_iterator):
    if i>10000:
      break
    calibration_texts.append(text_item)

print(f"Number of calibration texts collected: {len(calibration_texts)}")

Number of calibration texts collected: 10001


In [16]:
import pandas as pd

patients = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/apollo2v3/csv/patients.csv")
conditions = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/apollo2v3/csv/conditions.csv")
medications = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/apollo2v3/csv/medications.csv")
encounters = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/apollo2v3/csv/encounters.csv")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [17]:
cond_map = conditions.groupby("PATIENT")["DESCRIPTION"].apply(list).to_dict()
med_map = medications.groupby("PATIENT")["DESCRIPTION"].apply(list).to_dict()
enc_map = encounters.groupby("PATIENT")["DESCRIPTION"].apply(list).to_dict()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [18]:
import random

def generate_tasks(pid, patient):
    conds = cond_map.get(pid, [])
    meds = med_map.get(pid, [])
    encs = enc_map.get(pid, [])

    age = 2024 - int(patient["BIRTHDATE"][:4])
    gender = patient["GENDER"]

    base_text = f"{age}-year-old {gender}. Conditions: {', '.join(conds[:5])}."

    tasks = []

    # Task 1: Summary
    tasks.append({
        "instruction": "Summarize the patient's medical condition",
        "input": base_text,
        "output": f"The patient has {', '.join(conds[:3])}."
    })

    # Task 2: Diagnosis
    tasks.append({
        "instruction": "What conditions does this patient have?",
        "input": base_text,
        "output": ", ".join(conds[:5])
    })

    # Task 3: Medication reasoning
    if meds:
        tasks.append({
            "instruction": "What medications is the patient taking?",
            "input": base_text,
            "output": ", ".join(meds[:5])
        })

    # Task 4: Encounter reasoning
    if encs:
        tasks.append({
            "instruction": "What type of medical visits has the patient had?",
            "input": base_text,
            "output": ", ".join(encs[:3])
        })

    # Task 5: Clinical note
    tasks.append({
        "instruction": "Write a brief clinical note",
        "input": base_text,
        "output": f"{age}-year-old {gender} with history of {', '.join(conds[:3])}."
    })

    return tasks

In [19]:
dataset = []

for _, patient in patients.iterrows():
    pid = patient["Id"]

    tasks = generate_tasks(pid, patient)

    # repeat with variation
    for _ in range(10):  # key multiplier
        for t in tasks:
            # slight variation
            t_copy = t.copy()
            if random.random() > 0.5:
                t_copy["instruction"] += " Please be concise."

            dataset.append(t_copy)

# limit to ~200K
ds = dataset[:200000]

print(len(ds))

180000


In [20]:
print(ds[100])
prompt = '''{instruction}. The basic details of patient: {input} Assistant: {output}
'''
calibration_texts=[]
for sample in ds:
  calibration_texts.append(prompt.format(instruction = sample['instruction'],input = sample['input'],output = sample['output']))

{'instruction': "Summarize the patient's medical condition Please be concise.", 'input': '41-year-old M. Conditions: Adolescent idiopathic scoliosis (disorder), Received higher education (finding), Prediabetes (finding), Anemia (disorder), Chronic intractable migraine without aura (disorder).', 'output': 'The patient has Adolescent idiopathic scoliosis (disorder), Received higher education (finding), Prediabetes (finding).'}


In [ ]:
import json
import os
'''
# Fix: The previous installation of onnxruntime-genai was cancelled,
# leading to ModuleNotFoundError. We also had persistent CUDA mismatch issues.
# To ensure a successful installation with Colab-compatible CUDA,
# we will uninstall all onnxruntime-related packages and then
# install a specific version of onnxruntime-genai-cuda (0.11.0)
# which is often compatible with Colab's CUDA 11.x/12.x environments.

# First, ensure a clean slate by uninstalling all onnxruntime-related packages.
# This prevents conflicts with existing or partially installed versions.
!pip uninstall -y onnxruntime onnxruntime-gpu onnxruntime-genai onnxruntime-genai-cuda

# Install onnxruntime-genai-cuda==0.11.0. This package bundles a compatible
# onnxruntime-gpu, resolving the CUDA library mismatch (libcudart.so.13)
# and ensuring the onnxruntime_genai modules are correctly installed.
!pip install onnxruntime-genai-cuda==0.11.0 --force-reinstall
'''
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer


model_path = '/content/Apollo2'
quant_path = '/content/awq-out'
output_path = '/content/onnx-out'
execution_provider = "cuda"

quant_config = {"zero_point": True,
                "q_group_size": 128,
                "w_bit": 4,
                "version": "GEMM"}

# Load model
#model = AutoAWQForCausalLM.from_pretrained(model_path, low_cpu_mem_usage=True, use_cache=False)
model = AutoAWQForCausalLM.from_pretrained(model_path)

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

# Quantize model
model.quantize(tokenizer,
               quant_config=quant_config,
               calib_data =calibration_texts,
               max_calib_seq_len = 512,
                max_calib_samples = 4000)
# Save quantized model
model.save_quantized(quant_path)
tokenizer.save_pretrained(quant_path)

print(f'Model is quantized and saved at "{quant_path}"')


AWQ:  89%|████████▉ | 25/28 [06:39<00:48, 16.08s/it]

In [ ]:
'''
# 1. Uninstall all conflicting onnxruntime, onnxruntime-genai, and transformers packages.
!pip uninstall -y onnxruntime onnxruntime-gpu onnxruntime-genai onnxruntime-genai-cuda

# 4. Install the base onnxruntime-genai package, which will now use the installed onnxruntime-gpu.
!pip install onnxruntime-genai==0.12.0
'''
# Verify installations (optional, but good for debugging)
import onnxruntime as ort
print(f"ONNX Runtime providers: {ort.get_available_providers()}")
import onnxruntime_genai as og
print(f"ONNX Runtime GenAI version: {og.__version__}")

# Original code to create ONNX model
from onnxruntime_genai.models.builder import create_model

model_name = None
input_folder = '/content/awq-out'
output_folder = '/content/onnx-out'
precision = "int4"
execution_provider = "cuda"
cache_dir = '/content/temp'

create_model(model_name, input_folder, output_folder, precision, execution_provider, cache_dir)

In [ ]:
!cp -r "onnx-out" "/content/drive/MyDrive/Colab Notebooks/apollo2v3"
#!pip freeze > requirements.txt

In [ ]:
import time
import json
import numpy as np

import onnxruntime_genai as og

from datasets import load_dataset
from tqdm import tqdm

# =========================
# CONFIG
# =========================

#MODEL_PATH = '/content/drive/MyDrive/Colab Notebooks/apollo2-onnx/onnx-out/'
MODEL_PATH = './onnx-out/'

EVAL_SAMPLES = 30
MAX_LENGTH = 512

# =========================
# LOAD MODEL
# =========================

print("Loading ONNX model...")
config = og.Config(MODEL_PATH)
config.clear_providers()
config.append_provider("AZURE")
model = og.Model(config)
print("Model loaded")
#model = og.Model(MODEL_PATH)

tokenizer = og.Tokenizer(model)

# =========================
# GENERATION FUNCTION
# =========================

def generate_answer(prompt):

    tokens = tokenizer.encode(prompt)
    input_length = len(tokens)

    # Create fresh params per request
    params = og.GeneratorParams(model)

    params.set_search_options(
        max_length=input_length + MAX_LENGTH,
        temperature=0.0,
        do_sample=False
    )

    generator = og.Generator(model, params)


    start = time.time()
    generator.append_tokens(tokens)

    while not generator.is_done():
            generator.generate_next_token()

    end = time.time()

    output_tokens = generator.get_sequence(0)

    text = tokenizer.decode(output_tokens)

    answer = text[len(prompt):].strip()

    latency = end - start

    #token_count = len(output_tokens)

    return answer, latency, input_length

def _process_logits_chunk(
    logits_list: list[np.ndarray],
    targets_list: list[int],
) -> tuple[float, int]:
    """Calculates NLL sum and count for a chunk of logits and targets."""
    chunk_logits = np.stack(logits_list)
    chunk_targets = np.array(targets_list)

    c = np.max(chunk_logits, axis=1, keepdims=True)
    lse = c + np.log(np.sum(np.exp(chunk_logits - c), axis=1, keepdims=True))

    row_indices = np.arange(len(chunk_targets))
    target_logits = chunk_logits[row_indices, chunk_targets]

    # Sum of NLL for this chunk
    # NLL = lse - target_logits
    chunk_nll_sum = np.sum(lse.squeeze() - target_logits)

    return float(chunk_nll_sum), len(chunk_targets)


def calculate_perplexity_onnxruntime_genai(text: str) -> float:
    """Calculates perplexity using optimized NumPy operations."""
    input_ids = tokenizer.encode(text)
    generator = og.Generator(model, og.GeneratorParams(model))

    input_ids_int32 = (
        input_ids.astype(np.int32) if input_ids.dtype != np.int32 else input_ids
    )

    logits_list = []
    targets_list = []
    append_tokens = generator.append_tokens
    get_logits = generator.get_logits

    nll_sum = 0.0
    count = 0
    chunk_size = 128

    # Iterate over input tokens to predict the next token
    # i goes from 0 to len-2.
    # input: input_ids_int32[i]
    # target: input_ids_int32[i+1]
    for i in range(len(input_ids_int32) - 1):
        append_tokens(input_ids_int32[i : i + 1])
        logits_list.append(get_logits()[0, 0, :])
        targets_list.append(int(input_ids_int32[i + 1]))

        if len(logits_list) >= chunk_size:
            # Process chunk
            chunk_nll_sum, chunk_count = _process_logits_chunk(
                logits_list,
                targets_list,
            )
            nll_sum += chunk_nll_sum
            count += chunk_count

            logits_list = []
            targets_list = []

    # Process remaining logits
    if logits_list:
        chunk_nll_sum, chunk_count = _process_logits_chunk(logits_list, targets_list)
        nll_sum += chunk_nll_sum
        count += chunk_count

    if count == 0:
        return 0.0

    return float(np.exp(nll_sum / count))

# =========================
# PUBMEDQA BENCHMARK
# =========================

print("\nRunning PubMedQA...")

pubmedqa = load_dataset(
    "openlifescienceai/pubmedqa",
    split="test"
)
results=np.zeros((500,3))
correct = 0
total = 0
for sample in pubmedqa.select(range(EVAL_SAMPLES)):

    context = " ".join(
        sample["data"]["Context"]
    )

    question = sample["data"]["Question"]

    prompt = f"""
Context:
{context}

Question:
{question}

Answer yes, no, or maybe.

Answer:
"""
    '''
    answer_text, latency, tokens = generate_answer(prompt)

    prediction = answer_text.lower()

    gold = sample["data"]["Correct Answer"].lower()

    if gold in prediction:
        correct += 1
        results[total,0]=1
    '''
    ppl = calculate_perplexity_onnxruntime_genai(prompt)

    total += 1
    print(ppl)
    #print(total,",",correct,",",tokens,",",ppl)